# 🛰️ Inference Engine API Server (Colab GPU → your UI)

Runs the custom inference engine on this Colab GPU and exposes it as an
**OpenAI-compatible HTTP API** through a public HTTPS tunnel. Your separate UI
(`ui/` folder, hosted anywhere) connects to the printed URL.

**One-time setup**
1. On your PC: `python scripts/pack_for_colab.py` → upload `dist/custom_inference_engine.zip` to the root of **My Drive**.
2. *(Recommended)* In Colab's 🔑 **Secrets** panel add `ENGINE_API_KEY` (any long random string) so the key stays the same across sessions.

**Every session:** `Runtime → Change runtime type → A100 GPU`, then **Run all**.
Weights are downloaded to Drive the first time only; later sessions load straight from Drive.

## 1. GPU check + mount Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the engine code from Drive

In [ ]:
import os, shutil, zipfile

DRIVE = '/content/drive/MyDrive'
PROJECT_ZIP = f'{DRIVE}/custom_inference_engine.zip'     # from scripts/pack_for_colab.py
PROJECT_FOLDER = f'{DRIVE}/custom_inference_engine'       # alternative: the whole folder synced to Drive
WORKDIR = '/content/custom_inference_engine'

shutil.rmtree(WORKDIR, ignore_errors=True)
if os.path.isfile(PROJECT_ZIP):
    with zipfile.ZipFile(PROJECT_ZIP) as zf:
        zf.extractall('/content')
    print(f'Extracted {PROJECT_ZIP}')
elif os.path.isdir(PROJECT_FOLDER):
    shutil.copytree(PROJECT_FOLDER, WORKDIR, ignore=shutil.ignore_patterns('__pycache__', '*.safetensors'))
    print(f'Copied {PROJECT_FOLDER}')
else:
    raise FileNotFoundError('Upload custom_inference_engine.zip (or the folder) to the root of My Drive first.')

os.chdir(WORKDIR)
print(sorted(os.listdir('.')))

## 3. Install server dependencies

In [ ]:
!pip install -q -r server/requirements.txt

## 4. Launch server + tunnel

* `MODEL_REPO`: any Qwen2/Qwen2.5 dense checkpoint works (default **Qwen2.5-Coder-0.5B-Instruct**).
* `MODEL_DIR`: where weights live on Drive. If the folder is complete, **no download happens**.
  Already have the weights on your PC? Upload that folder (with `config.json`, `model.safetensors`, `tokenizer.json`, …) to this path.
* `TUNNEL`: `"cloudflare"` (no account) or `"ngrok"` (needs `NGROK_TOKEN` secret, stable domain on paid plans).

In [ ]:
import sys, importlib
sys.path.insert(0, WORKDIR)
from server import colab_launcher
importlib.reload(colab_launcher)

MODEL_REPO = 'Qwen/Qwen2.5-Coder-0.5B-Instruct'
MODEL_DIR = f'{DRIVE}/llm_models/qwen2.5-coder-0.5b-instruct'
TUNNEL = 'cloudflare'

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

info = colab_launcher.launch(
    api_key=secret('ENGINE_API_KEY'),          # None -> random key printed below
    model_repo=MODEL_REPO,
    model_dir=MODEL_DIR,
    tunnel=TUNNEL,
    ngrok_token=secret('NGROK_TOKEN'),
    engine_env={
        'ENGINE_MAX_MODEL_LEN': '8192',
        'ENGINE_NUM_BLOCKS': '4096',
        'ENGINE_MAX_BATCH_SIZE': '32',
        'ENGINE_TEMPERATURE': '0.0',            # 0 = greedy; server-wide setting
    },
)

## 5. Smoke test (optional)

In [ ]:
import json, urllib.request
req = urllib.request.Request(
    'http://127.0.0.1:8000/v1/chat/completions',
    data=json.dumps({'messages': [{'role': 'user', 'content': 'Write a Python function that reverses a string.'}],
                     'max_tokens': 128}).encode(),
    headers={'Content-Type': 'application/json', 'Authorization': f"Bearer {info['api_key']}"},
)
resp = json.loads(urllib.request.urlopen(req).read())
print(resp['choices'][0]['message']['content'])
print(resp['metrics'])

## 6. Open the UI

Open the `ui/` folder from your PC (any static host, or `python -m http.server 5173` inside `ui/`)
and paste the URL + key into **Settings**. Or use the deep link printed below.

Keep this tab open: the server lives as long as the Colab runtime does.

In [ ]:
print('UI deep link (replace the host with wherever your UI is served):')
print(f"http://localhost:5173/?api={info['url']}&key={info['api_key']}")

## Utilities

In [ ]:
# Server logs (engine load, CUDA graph capture, requests)
colab_launcher.server_logs(80)

In [ ]:
# Stop server + tunnel
colab_launcher.stop_all()